# Treinamento de um agente de Blackjack com Q-Learning

## Objetivo
Este notebook implementa o treinamento de um agente para o jogo Blackjack utilizando o algoritmo **Q-Learning**.

O objetivo é permitir que o agente aprenda, por interação com o ambiente, quais ações tendem a produzir melhores resultados ao longo do tempo.

## Papel desta etapa no projeto

Após a implementação do ambiente e da comparação entre políticas simples, esta etapa introduz uma política aprendida com **aprendizado por reforço**.

A proposta é verificar se um agente treinado com experiência acumulada consegue superar estratégias de referência, como a política aleatória e a política básica.

## Reutilização do ambiente

A lógica do jogo Blackjack não é redefinida neste notebook.

As funções responsáveis pela dinâmica do ambiente foram centralizadas no arquivo `blackjack_env.py`, incluindo:
- inicialização da partida;
- representação do estado;
- execução da ação no ambiente;
- cálculo do desfecho final.

Essa organização evita duplicação de código e torna o projeto mais modular e reutilizável.

## Importações e configuração inicial

Nesta seção são importadas as bibliotecas necessárias, as funções do ambiente e as funções auxiliares de salvamento dos resultados.
Também é definida uma semente aleatória para tornar o experimento reproduzível.

In [17]:
import random
import pandas as pd

from utils_io import salvar_df

In [2]:
random.seed(42)

In [3]:
from blackjack_env import valor_mao, estado_jogo, iniciar_partida, step_blackjack

## Funções auxiliares do algoritmo

As funções abaixo dão suporte ao Q-Learning:

- garantir que um estado exista na Q-Table;
- selecionar a melhor ação conhecida;
- aplicar a estratégia **epsilon-greedy**, que equilibra exploração e aproveitamento.

In [4]:
ACOES = ["hit", "stick"]

def garantir_estado(q_table, estado):
    if estado not in q_table:
        q_table[estado] = {"hit": 0.0, "stick": 0.0}

def melhor_acao(q_table, estado):
    garantir_estado(q_table, estado)

    q_hit = q_table[estado]["hit"]
    q_stick = q_table[estado]["stick"]

    if q_hit > q_stick:
        return "hit"
    elif q_stick > q_hit:
        return "stick"
    else:
        return random.choice(ACOES)

def escolher_acao_epsilon_greedy(q_table, estado, epsilon):
    garantir_estado(q_table, estado)

    if random.random() < epsilon:
        return random.choice(ACOES)
    return melhor_acao(q_table, estado)

## Hiperparâmetros do treinamento

O treinamento do agente depende de alguns parâmetros principais:

- `alpha`: taxa de aprendizado;
- `gamma`: fator de desconto;
- `epsilon_inicial`: probabilidade inicial de explorar ações aleatórias;
- `epsilon_min`: limite inferior da exploração;
- `epsilon_decay`: taxa de redução do epsilon ao longo dos episódios.

Também são definidos o número de episódios de treinamento e o número de episódios de avaliação.

In [21]:
alpha = 0.1
gamma = 1.0
epsilon_inicial = 1.0
epsilon_min = 0.05
epsilon_decay = 0.9995

n_episodios_treinamento = 100000
n_episodios_avaliacao = 100000

## Treinamento do agente

Nesta etapa, o agente interage repetidamente com o ambiente.

Em cada episódio:
1. o estado inicial é observado;
2. uma ação é escolhida com estratégia epsilon-greedy;
3. o ambiente retorna o próximo estado e a recompensa;
4. a Q-Table é atualizada com base na equação do Q-Learning.

Ao longo do treinamento, o valor de `epsilon` é reduzido gradualmente, permitindo que o agente explore mais no início e aproveite mais o conhecimento adquirido nas fases posteriores.

In [6]:
q_table = {}
historico_treinamento = []

epsilon = epsilon_inicial

for episodio in range(n_episodios_treinamento):
    mao_jogador, mao_dealer = iniciar_partida()
    estado = estado_jogo(mao_jogador, mao_dealer[0])

    done = False
    passos = 0
    recompensa_final = 0
    resultado_final = "em_andamento"

    while not done:
        garantir_estado(q_table, estado)

        acao = escolher_acao_epsilon_greedy(q_table, estado, epsilon)

        proximo_estado, recompensa, done, resultado, mao_jogador, mao_dealer = step_blackjack(
            mao_jogador, mao_dealer, acao
        )

        q_atual = q_table[estado][acao]

        if done:
            target = recompensa
        else:
            garantir_estado(q_table, proximo_estado)
            target = recompensa + gamma * max(q_table[proximo_estado].values())

        q_table[estado][acao] = q_atual + alpha * (target - q_atual)

        estado = proximo_estado if proximo_estado is not None else estado
        passos += 1
        recompensa_final = recompensa
        resultado_final = resultado

    historico_treinamento.append({
        "id_episodio": episodio,
        "resultado": resultado_final,
        "recompensa_total": recompensa_final,
        "qtd_passos": passos,
        "epsilon": epsilon,
        "mao_jogador_final": str(mao_jogador),
        "mao_dealer_final": str(mao_dealer)
    })

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

## Histórico do treinamento

As informações do treinamento são registradas em uma base auxiliar para acompanhar a evolução do agente ao longo dos episódios, incluindo:

- recompensa total por episódio;
- número de passos;
- resultado final;
- valor atual de epsilon.

In [7]:
df_treinamento = pd.DataFrame(historico_treinamento)
df_treinamento.head()

,id_episodio,resultado,recompensa_total,qtd_passos,epsilon,mao_jogador_final,mao_dealer_final
0,0,derrota,-1,2,1.000000,"[10, 2, 3, 10]","[1, 10]"
1,1,empate,0,2,0.999500,"[7, 1, 9]","[1, 2, 4]"
2,2,derrota,-1,1,0.999000,"[8, 10, 10]","[5, 10]"
3,3,derrota,-1,1,0.998501,"[7, 6]","[5, 3, 2, 2, 7]"
4,4,derrota,-1,2,0.998001,"[2, 6, 10, 7]","[6, 10]"


## Estrutura final da Q-Table

Após o treinamento, a Q-Table é convertida em um DataFrame para facilitar a inspeção dos estados aprendidos e da ação preferida em cada um deles.

In [8]:
linhas_qtable = []

for estado, valores in q_table.items():
    soma_jogador, carta_dealer, as_utilizavel = estado

    q_hit = valores["hit"]
    q_stick = valores["stick"]

    if q_hit > q_stick:
        melhor = "hit"
    elif q_stick > q_hit:
        melhor = "stick"
    else:
        melhor = "empate"

    linhas_qtable.append({
        "soma_jogador": soma_jogador,
        "carta_dealer": carta_dealer,
        "as_utilizavel": as_utilizavel,
        "q_hit": q_hit,
        "q_stick": q_stick,
        "melhor_acao": melhor
    })

df_qtable = pd.DataFrame(linhas_qtable).sort_values(
    by=["soma_jogador", "carta_dealer", "as_utilizavel"]
).reset_index(drop=True)

df_qtable.head(20)

,soma_jogador,carta_dealer,as_utilizavel,q_hit,q_stick,melhor_acao
0,4,1,0,-0.280786,-0.271000,stick
1,4,2,0,-0.063072,-0.100000,hit
2,4,3,0,-0.070284,-0.271000,hit
3,4,4,0,-0.061236,-0.100000,hit
4,4,5,0,-0.009023,-0.100000,hit
5,4,6,0,0.036668,-0.100000,hit
6,4,7,0,-0.040205,-0.100000,hit
7,4,8,0,-0.018391,-0.100000,hit
8,4,9,0,-0.164974,-0.191539,hit
9,4,10,0,-0.340179,-0.409510,hit


## Avaliação da política aprendida

Depois do treinamento, a Q-Table é utilizada para definir a política do agente.

Nessa etapa, o agente deixa de explorar aleatoriamente e passa a selecionar, para cada estado, a ação com maior valor estimado. Isso permite medir o desempenho real da política aprendida.

In [9]:
def politica_qlearning(estado):
    garantir_estado(q_table, estado)
    return melhor_acao(q_table, estado)

In [10]:
def avaliar_politica_qlearning(n_episodios=5000):
    resultados = []

    for episodio in range(n_episodios):
        mao_jogador, mao_dealer = iniciar_partida()
        carta_visivel_dealer = mao_dealer[0]

        done = False
        passos = 0
        resultado_final = "em_andamento"
        recompensa_final = 0

        while not done:
            estado = estado_jogo(mao_jogador, carta_visivel_dealer)
            acao = politica_qlearning(estado)

            proximo_estado, recompensa, done, resultado, mao_jogador, mao_dealer = step_blackjack(
                mao_jogador, mao_dealer, acao
            )

            passos += 1
            recompensa_final = recompensa
            resultado_final = resultado

        resultados.append({
            "id_episodio": episodio,
            "politica": "qlearning",
            "resultado": resultado_final,
            "recompensa": recompensa_final,
            "total_jogador": valor_mao(mao_jogador),
            "total_dealer": valor_mao(mao_dealer),
            "dealer_carta_visivel": carta_visivel_dealer,
            "qtd_passos": passos,
            "mao_jogador": str(mao_jogador),
            "mao_dealer": str(mao_dealer)
        })

    return pd.DataFrame(resultados)

df_resultados_qlearning = avaliar_politica_qlearning(n_episodios=n_episodios_avaliacao)
df_resultados_qlearning.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,qtd_passos,mao_jogador,mao_dealer
0,0,qlearning,empate,0,20,20,1,1,"[10, 10]","[1, 9]"
1,1,qlearning,derrota,-1,15,21,4,1,"[5, 10]","[4, 3, 4, 5, 5]"
2,2,qlearning,derrota,-1,18,21,10,1,"[10, 8]","[10, 3, 8]"
3,3,qlearning,derrota,-1,16,19,6,1,"[6, 10]","[6, 6, 7]"
4,4,qlearning,derrota,-1,18,21,10,1,"[8, 10]","[10, 1]"


## Métricas de avaliação

A política treinada é avaliada com base em indicadores como:

- recompensa média;
- taxa de vitória;
- taxa de derrota;
- taxa de empate.

Essas métricas serão utilizadas posteriormente para comparar o Q-Learning com as políticas de referência.

In [11]:
taxa_vitoria = (df_resultados_qlearning["resultado"] == "vitoria").mean()
taxa_derrota = (df_resultados_qlearning["resultado"] == "derrota").mean()
taxa_empate = (df_resultados_qlearning["resultado"] == "empate").mean()

df_metricas_qlearning = pd.DataFrame([{
    "politica": "qlearning",
    "episodios_treinamento": n_episodios_treinamento,
    "episodios_avaliacao": n_episodios_avaliacao,
    "alpha": alpha,
    "gamma": gamma,
    "epsilon_inicial": epsilon_inicial,
    "epsilon_final": epsilon,
    "epsilon_min": epsilon_min,
    "epsilon_decay": epsilon_decay,
    "recompensa_media_treinamento": df_treinamento["recompensa_total"].mean(),
    "recompensa_media_avaliacao": df_resultados_qlearning["recompensa"].mean(),
    "taxa_vitoria_avaliacao": taxa_vitoria,
    "taxa_derrota_avaliacao": taxa_derrota,
    "taxa_empate_avaliacao": taxa_empate,
    "passos_medios_avaliacao": df_resultados_qlearning["qtd_passos"].mean()
}])

df_metricas_qlearning

,politica,episodios_treinamento,episodios_avaliacao,alpha,gamma,epsilon_inicial,epsilon_final,epsilon_min,epsilon_decay,recompensa_media_treinamento,recompensa_media_avaliacao,taxa_vitoria_avaliacao,taxa_derrota_avaliacao,taxa_empate_avaliacao,passos_medios_avaliacao
0,qlearning,20000,5000,0.1,1.0,1.0,0.05,0.05,0.9995,-0.144,-0.0496,0.4304,0.48,0.0896,1.5654


## Métricas de avaliação

A política treinada é avaliada com base em indicadores como:

- recompensa média;
- taxa de vitória;
- taxa de derrota;
- taxa de empate.

Essas métricas serão utilizadas posteriormente para comparar o Q-Learning com as políticas de referência.

In [12]:
df_resultados_qlearning["resultado"].value_counts()

resultado
derrota    2400
vitoria    2152
empate      448
Name: count, dtype: int64

## Resumo consolidado do treinamento e da avaliação

A tabela abaixo reúne os principais hiperparâmetros e métricas finais do experimento, permitindo registrar de forma objetiva a configuração utilizada e os resultados obtidos.

In [13]:
df_metricas_qlearning

,politica,episodios_treinamento,episodios_avaliacao,alpha,gamma,epsilon_inicial,epsilon_final,epsilon_min,epsilon_decay,recompensa_media_treinamento,recompensa_media_avaliacao,taxa_vitoria_avaliacao,taxa_derrota_avaliacao,taxa_empate_avaliacao,passos_medios_avaliacao
0,qlearning,20000,5000,0.1,1.0,1.0,0.05,0.05,0.9995,-0.144,-0.0496,0.4304,0.48,0.0896,1.5654


## Amostra da Q-Table aprendida

A visualização parcial da Q-Table permite inspecionar como o agente passou a valorizar as ações `hit` e `stick` em diferentes estados do jogo.

In [14]:
df_qtable.head(30)

,soma_jogador,carta_dealer,as_utilizavel,q_hit,q_stick,melhor_acao
0,4,1,0,-0.280786,-0.271000,stick
1,4,2,0,-0.063072,-0.100000,hit
2,4,3,0,-0.070284,-0.271000,hit
3,4,4,0,-0.061236,-0.100000,hit
4,4,5,0,-0.009023,-0.100000,hit
5,4,6,0,0.036668,-0.100000,hit
6,4,7,0,-0.040205,-0.100000,hit
7,4,8,0,-0.018391,-0.100000,hit
8,4,9,0,-0.164974,-0.191539,hit
9,4,10,0,-0.340179,-0.409510,hit


## Persistência dos artefatos

Ao final do processo, são salvos:

- a Q-Table aprendida;
- o histórico do treinamento;
- os resultados da avaliação;
- as métricas consolidadas.

Esses arquivos serão reutilizados nas etapas seguintes de avaliação comparativa e análise exploratória.

In [15]:
salvar_df(df_qtable, "blackjack_q_table", pasta="modelos")
salvar_df(df_treinamento, "blackjack_treinamento_qlearning", pasta="dados")
salvar_df(df_resultados_qlearning, "blackjack_resultados_qlearning", pasta="dados")
salvar_df(df_metricas_qlearning, "blackjack_metricas_qlearning", pasta="resultados")

Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\modelos\blackjack_q_table.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_treinamento_qlearning.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_resultados_qlearning.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_metricas_qlearning.csv


In [ ]:
df_metricas_qlearning.head()

In [ ]:
df_qtable.head(20)

## Conclusão

Nesta etapa foi treinado um agente de Blackjack com Q-Learning a partir da interação direta com o ambiente.

O experimento permite observar como o agente evolui ao longo do treinamento e gera uma política aprendida que poderá ser comparada às estratégias de referência definidas anteriormente.

Com isso, o projeto avança da comparação entre regras fixas para uma abordagem baseada em aprendizado por experiência.